# 01 — Data Understanding

First pass over the Stack Overflow Developer Survey 2025 raw data: load both files, cross-reference the schema, and figure out exactly what we're working with before we clean anything in Notebook 2.

## Setup

Put `survey_results_public.csv` and `survey_results_schema.csv` in the `data/` folder before running this (rename them if you downloaded them as `results.txt`/`schema.txt` from the GitHub archive — same content, just cleaner names to keep in the repo).

**Important:** load with the default engine, not `engine="pyarrow"` — a few free-text fields contain embedded line breaks that trip up the PyArrow parser.

In [14]:
import pandas as pd

DATA_DIR = "C:\\Users\\ASUS\\Desktop\\neural_career_advisor"
df = pd.read_csv(f"{DATA_DIR}/results.txt", low_memory=False)
schema = pd.read_csv(f"{DATA_DIR}/schema.txt")

print("Responses:", df.shape[0])
print("Columns:", df.shape[1])
print("Memory usage:", round(df.memory_usage(deep=True).sum() / 1e6, 1), "MB")

Responses: 49191
Columns: 172
Memory usage: 375.2 MB


## Cross-referencing the schema

The schema file documents each *conceptual* question, but several question types — anything with a "have worked with / want to work with / admired" split, like languages or databases — expand into multiple real columns that don't appear in the schema under that exact name. Worth mapping this explicitly so we don't lose track of what any column actually means.

In [15]:
data_cols = set(df.columns)
schema_qnames = set(schema["qname"].dropna())

only_in_data = sorted(data_cols - schema_qnames)
only_in_schema = sorted(schema_qnames - data_cols)

print(f"Columns in data but not named directly in schema: {len(only_in_data)}")
print(only_in_data)
print()
print(f"Base questions in schema that expand into multiple columns: {len(only_in_schema)}")
print(only_in_schema)

Columns in data but not named directly in schema: 46
['AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat agree', 'AIAgentChallengesSomewhat disagree', 'AIAgentChallengesStrongly agree', 'AIAgentChallengesStrongly disagree', 'AIAgentImpactNeutral', 'AIAgentImpactSomewhat agree', 'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree', 'AIAgentImpactStrongly disagree', 'AIModelsAdmired', 'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith', 'AIToolCurrently mostly AI', 'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task", 'AIToolPlan to mostly use AI', 'AIToolPlan to partially use AI', 'CommPlatformAdmired', 'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'ConvertedCompYearly', 'DatabaseAdmired', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'DevEnvsAdmired', 'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith', 'LanguageAdmired', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'OfficeStackAsyncAdmired', 'OfficeStackAsyncHaveWorkedWith', 

Most of the 46 are the expanded `HaveWorkedWith` / `WantToWorkWith` / `Admired` triplets for Language, Database, Platform, Webframe, DevEnvs, OfficeStackAsync, CommPlatform, SOTags, and AIModels — each base question in the schema (e.g. `Language`) becomes three real columns. A handful of others are administrative (`ResponseId`) or derived (`ConvertedCompYearly` — the pre-converted-to-USD salary field, which is what we'll actually use later, not the raw `CompTotal`).

## Classifying every column

Three buckets matter for what comes next: numeric, single-select categorical, and multi-select (semicolon-delimited).

**Note:** this pandas version (3.0+) uses a native `str` dtype for text columns instead of the older `object` dtype. If you're following an older pandas tutorial, check for both — code that only checks `dtype == 'object'` will silently miss every string column here rather than error, which is an easy bug to miss.

In [16]:
profile = []
for col in df.columns:
    s = df[col]
    dtype = str(s.dtype)
    is_multiselect = False
    if dtype in ("str", "object"):
        sample = s.dropna().astype(str).head(1000)
        is_multiselect = bool(sample.str.contains(";", regex=False).any())
    profile.append({
        "column": col,
        "dtype": dtype,
        "missing_pct": round(s.isna().mean() * 100, 1),
        "n_unique": s.nunique(dropna=True),
        "multi_select": is_multiselect,
    })

col_profile = pd.DataFrame(profile)
col_profile.to_csv(f"{DATA_DIR}/column_profile.csv", index=False)

print("Multi-select columns:", int(col_profile["multi_select"].sum()))
print("Numeric columns:", int(col_profile["dtype"].isin(["int64", "float64"]).sum()))
n_single = ((col_profile["dtype"].isin(["str", "object"])) & (~col_profile["multi_select"]) & (col_profile["n_unique"] <= 25)).sum()
print("Single-select categorical (<=25 unique values):", int(n_single))

Multi-select columns: 75
Numeric columns: 53
Single-select categorical (<=25 unique values): 32


## Missingness — the columns that matter most for us

Salary and a handful of optional fields have heavy non-response. Worth knowing the real numbers before we build anything on top of them — any salary fact we generate later needs to report its sample size alongside the figure, not just the number on its own.

In [17]:
key_cols = ["ConvertedCompYearly", "CompTotal", "DevType", "Country", "YearsCode",
            "WorkExp", "JobSat", "RemoteWork", "OrgSize", "EdLevel"]
print(col_profile[col_profile["column"].isin(key_cols)][["column", "dtype", "missing_pct", "n_unique"]].to_string(index=False))

             column   dtype  missing_pct  n_unique
            EdLevel  object          2.1         8
            WorkExp float64         12.8        72
          YearsCode float64         12.5        78
            DevType  object         11.2        32
            OrgSize  object         30.5         9
         RemoteWork  object         31.3         5
            Country  object         28.0       177
          CompTotal float64         49.5      2678
ConvertedCompYearly float64         51.3      6237
             JobSat float64         45.8        11


## What we now know

- **49,191 responses × 172 columns**, ~375MB in memory — matches the official survey scale closely
- 75 columns are multi-select (semicolon-delimited) — these need the parser we'll build in Notebook 2
- 53 numeric, 32 single-select categorical, the rest free text or high-cardinality write-ins
- `ConvertedCompYearly` (our salary target) is missing for roughly half of respondents — expect to report sample sizes alongside any salary fact
- `Country` has exactly 177 unique values, matching Stack Overflow's own reported count — good sign the data's intact

**Next:** Notebook 2 — cleaning & EDA, starting with the multi-select parser.